# Compute estimate

Per-step cost of the four models section 29.1 requires, measured on the target machine. These
numbers are what the proposal's compute section reports.

In [1]:
import sys
import time

sys.path.insert(0, "..")

import torch
from torch import nn
from torchvision.models.segmentation import deeplabv3_mobilenet_v3_large
from transformers import AutoModel

from shared import io as shared_io
from shared.device import (
    count_parameters, describe_hardware, peak_memory, reset_peak_memory, select_device, synchronize,
)
from src.interfaces import NUM_CLASSES
from src.utils import load_config, resolve_path

config = load_config(resolve_path("configs/base.yaml"))
device = select_device()
size = tuple(config["input"]["size"])
batch_size = config["loader"]["batch_size"]
TEXT_MODEL = "distilbert-base-uncased"
# MPS has no scaled-dot-product kernel that supports dropout, so attention falls back to the
# eager path there; CUDA keeps the fused one, which is why the two backends are not comparable.
ATTENTION = "eager" if device.type == "mps" else "sdpa"
print(describe_hardware(), "|", size, "| batch", batch_size, "| attention", ATTENTION)

mps: Apple M4 Pro, 14 cores, 48 GB RAM | (320, 320) | batch 8 | attention eager


## The four models

Each takes the same three arguments, so a unimodal baseline drops whichever one it does not
read and all four share one training path.

In [2]:
class VisionBranch(nn.Module):
    """DeepLabV3 re-headed for one foreground class."""

    def __init__(self) -> None:
        super().__init__()
        # The published weights were trained with the auxiliary head, so it must be built and
        # then dropped; training it would cost time for an output no loss reads.
        self.network = deeplabv3_mobilenet_v3_large(weights="DEFAULT", aux_loss=True)
        self.network.aux_classifier = None
        head = self.network.classifier[-1]
        self.network.classifier[-1] = nn.Conv2d(head.in_channels, NUM_CLASSES, 1)

    def forward(self, images, tokens=None, attention=None):
        return self.network(images)["out"]


class TextBranch(nn.Module):
    """Predicts a mask from the expression alone: a spatial prior, no pixels read."""

    def __init__(self, size) -> None:
        super().__init__()
        self.encoder = AutoModel.from_pretrained(TEXT_MODEL, attn_implementation=ATTENTION)
        self.head = nn.Linear(self.encoder.config.hidden_size, 32 * 32 * NUM_CLASSES)
        self.size = size

    def pool(self, tokens, attention):
        states = self.encoder(input_ids=tokens, attention_mask=attention.long()).last_hidden_state
        weights = attention.unsqueeze(-1).float()
        return (states * weights).sum(1) / weights.sum(1).clamp(min=1)

    def forward(self, images, tokens, attention):
        coarse = self.head(self.pool(tokens, attention)).view(-1, NUM_CLASSES, 32, 32)
        return nn.functional.interpolate(coarse, self.size, mode="bilinear", align_corners=False)

In [3]:
class SimpleFusion(nn.Module):
    """Late fusion: one pooled sentence vector broadcast onto the feature map."""

    def __init__(self, frozen: bool = True) -> None:
        super().__init__()
        self.vision = deeplabv3_mobilenet_v3_large(weights="DEFAULT", aux_loss=True).backbone
        self.text = AutoModel.from_pretrained(TEXT_MODEL, attn_implementation=ATTENTION)
        if frozen:
            for parameter in self.text.parameters():
                parameter.requires_grad_(False)
        width = self.text.config.hidden_size
        self.project = nn.Conv2d(960 + width, 256, 1)
        self.head = nn.Sequential(nn.Conv2d(256, 256, 3, padding=1), nn.ReLU(inplace=True),
                                  nn.Conv2d(256, NUM_CLASSES, 1))

    def pool(self, tokens, attention):
        states = self.text(input_ids=tokens, attention_mask=attention.long()).last_hidden_state
        weights = attention.unsqueeze(-1).float()
        return (states * weights).sum(1) / weights.sum(1).clamp(min=1)

    def forward(self, images, tokens, attention):
        visual = self.vision(images)["out"]
        sentence = self.pool(tokens, attention)
        spread = sentence[:, :, None, None].expand(-1, -1, *visual.shape[-2:])
        fused = self.head(self.project(torch.cat([visual, spread], dim=1)))
        return nn.functional.interpolate(fused, images.shape[-2:], mode="bilinear", align_corners=False)


class CrossAttentionFusion(SimpleFusion):
    """Each visual position attends over the tokens, so a word can address a region."""

    def __init__(self) -> None:
        super().__init__(frozen=False)
        width = self.text.config.hidden_size
        self.to_query = nn.Conv2d(960, width, 1)
        self.attend = nn.MultiheadAttention(width, num_heads=8, batch_first=True)
        self.project = nn.Conv2d(960 + width, 256, 1)

    def forward(self, images, tokens, attention):
        visual = self.vision(images)["out"]
        states = self.text(input_ids=tokens, attention_mask=attention.long()).last_hidden_state
        n, _, h, w = visual.shape
        query = self.to_query(visual).flatten(2).transpose(1, 2)
        attended, _ = self.attend(query, states, states, key_padding_mask=~attention)
        attended = attended.transpose(1, 2).view(n, -1, h, w)
        fused = self.head(self.project(torch.cat([visual, attended], dim=1)))
        return nn.functional.interpolate(fused, images.shape[-2:], mode="bilinear", align_corners=False)

## Timing

A training step is forward, loss and backward; inference is forward only, without autograd.

In [4]:
criterion = nn.CrossEntropyLoss()
LENGTH = 8  # the 90th percentile expression is 6 tokens; 8 leaves room for the two special ones


def measure(build, warmup: int = 3, timed: int = 8) -> dict:
    model = build().to(device)
    images = torch.randn(batch_size, 3, *size, device=device)
    tokens = torch.randint(1, 1000, (batch_size, LENGTH), device=device)
    attention = torch.ones(batch_size, LENGTH, dtype=torch.bool, device=device)
    targets = torch.randint(0, NUM_CLASSES, (batch_size, *size), device=device)
    optimiser = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-4)

    reset_peak_memory(device)
    model.train()
    for index in range(warmup + timed):
        if index == warmup:
            synchronize(device)
            start = time.perf_counter()
        optimiser.zero_grad(set_to_none=True)
        criterion(model(images, tokens, attention), targets).backward()
        optimiser.step()
    synchronize(device)
    per_step = (time.perf_counter() - start) / timed

    model.eval()
    with torch.no_grad():
        for index in range(warmup + timed):
            if index == warmup:
                synchronize(device)
                start = time.perf_counter()
            model(images, tokens, attention)
        synchronize(device)
    per_pair = (time.perf_counter() - start) * 1000 / (timed * batch_size)

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = count_parameters(model)
    memory = peak_memory(device)
    del model, optimiser
    return {"parameters": total, "trainable": trainable, "seconds_per_step": per_step,
            "ms_per_pair": per_pair, "peak_memory_bytes": memory}

In [5]:
PAIRS_PER_EPOCH = 42404  # one expression per object, the training subset rule
EPOCHS = 20

builds = {
    "image only": VisionBranch,
    "text only": lambda: TextBranch(size),
    "simple fusion": lambda: SimpleFusion(frozen=True),
    "improved fusion": CrossAttentionFusion,
}

measured = {}
for name, build in builds.items():
    result = measure(build)
    steps = PAIRS_PER_EPOCH // batch_size
    result["steps_per_epoch"] = steps
    result["hours_per_epoch"] = steps * result["seconds_per_step"] / 3600
    result["hours_per_run"] = result["hours_per_epoch"] * EPOCHS
    measured[name] = result
    print(f"{name:<18}{result['trainable']:>12,} trainable  {result['seconds_per_step']:>6.3f} s/step  "
          f"{result['ms_per_pair']:>6.1f} ms/pair  {result['peak_memory_bytes'] / 1024 ** 3:>5.2f} GB  "
          f"{result['hours_per_run']:>6.2f} h/run")

image only          11,020,594 trainable   0.108 s/step     4.1 ms/pair   2.23 GB    3.17 h/run


W1114 01:24:32.519000 85114 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


text only           67,937,792 trainable   0.041 s/step     0.6 ms/pair   2.23 GB    1.20 h/run


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


simple fusion        4,005,170 trainable   0.101 s/step     3.4 ms/pair   2.24 GB    2.96 h/run


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


improved fusion     73,468,466 trainable   0.146 s/step     3.8 ms/pair   3.27 GB    4.30 h/run


## Planned budget

Three seeds for the two models the decision rule must separate, one for each baseline.

In [6]:
SEEDS = {"image only": 1, "text only": 1, "simple fusion": 3, "improved fusion": 3}

total = 0.0
for name, seeds in SEEDS.items():
    hours = seeds * measured[name]["hours_per_run"]
    total += hours
    print(f"{name:<18}{seeds} seed(s){hours:>8.2f} h")
print(f"{'total':<18}{'':>9}{total:>8.2f} h")

shared_io.write_json(resolve_path("results/compute.json"), {
    "hardware": describe_hardware(),
    "torch": torch.__version__,
    "input_size": list(size),
    "batch_size": batch_size,
    "text_model": TEXT_MODEL,
    "attention": ATTENTION,
    "pairs_per_epoch": PAIRS_PER_EPOCH,
    "epochs": EPOCHS,
    "configurations": measured,
    "budget_hours": total,
})

image only        1 seed(s)    3.17 h
text only         1 seed(s)    1.20 h
simple fusion     3 seed(s)    8.88 h
improved fusion   3 seed(s)   12.89 h
total                         26.14 h
